# Sentiment Classification using Classical Machine Learning and TF-IDF Features

This notebook builds a reproducible classical NLP baseline for IMDB sentiment classification and exports artifacts in a comparison-ready format for future transformer notebooks.

The workflow is intentionally engineering-oriented:
- Hugging Face IMDB loading
- train / validation / test splitting
- lightweight EDA
- TF-IDF feature engineering
- Multinomial Naive Bayes and Logistic Regression
- shared evaluation, error analysis, artifact saving, and comparison outputs

## Imports

In [ ]:
import json
import os
import random
import re
import warnings
from pathlib import Path
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
    }
)

print("Imports completed.")

## Config

In [ ]:
RANDOM_SEED = 42
PROJECT_TITLE = "Sentiment Classification using Classical Machine Learning and TF-IDF Features"
ARTIFACT_SCHEMA_VERSION = "classical_nlp_v1"

DATASET_NAME = "imdb"
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"
LABEL_NAMES = {0: "negative", 1: "positive"}

TRAIN_VALIDATION_SPLIT = 0.20

TFIDF_CONFIG = {
    "max_features": 50000,
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_df": 0.95,
    "stop_words": "english",
    "sublinear_tf": True,
}

NAIVE_BAYES_CONFIG = {
    "alpha": 1.0,
    "fit_prior": True,
}

LOGISTIC_REGRESSION_CONFIG = {
    "C": 2.0,
    "solver": "liblinear",
    "penalty": "l2",
    "class_weight": None,
    "max_iter": 1000,
    "tol": 1e-4,
}


def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def resolve_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "IMDB Dataset.csv").exists() or (candidate / DATASET_NAME).exists():
            return candidate
    return start


PROJECT_ROOT = resolve_project_root(Path.cwd().resolve())
EDA_DIR = PROJECT_ROOT / "eda_plots"
NAIVE_BAYES_DIR = PROJECT_ROOT / "naive_bayes_results"
LOGISTIC_REGRESSION_DIR = PROJECT_ROOT / "logistic_regression_results"

for directory in [EDA_DIR, NAIVE_BAYES_DIR, LOGISTIC_REGRESSION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

set_global_seed(RANDOM_SEED)

print(f"Project root: {PROJECT_ROOT}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Naive Bayes output dir: {NAIVE_BAYES_DIR}")
print(f"Logistic Regression output dir: {LOGISTIC_REGRESSION_DIR}")

## Dataset Loading

In [ ]:
def log_section(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def load_imdb_splits(seed: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, str]:
    try:
        print("Loading IMDB dataset from Hugging Face...")
        dataset = load_dataset("imdb")
        train_source = pd.DataFrame(dataset["train"])[[TEXT_COLUMN, LABEL_COLUMN]]
        test_df = pd.DataFrame(dataset["test"])[[TEXT_COLUMN, LABEL_COLUMN]]
        train_df, validation_df = train_test_split(
            train_source,
            test_size=TRAIN_VALIDATION_SPLIT,
            random_state=seed,
            stratify=train_source[LABEL_COLUMN],
        )
        source = "huggingface/imdb"
    except Exception as exc:
        fallback_path = PROJECT_ROOT / "IMDB Dataset.csv"
        if not fallback_path.exists():
            raise RuntimeError("Failed to load Hugging Face IMDB dataset and no local fallback was found.") from exc
        print(f"Hugging Face load failed, using local fallback: {fallback_path}")
        fallback = pd.read_csv(fallback_path)
        fallback = fallback.rename(columns={"review": TEXT_COLUMN, "sentiment": "sentiment"})
        fallback[LABEL_COLUMN] = fallback["sentiment"].map({"negative": 0, "positive": 1})
        fallback = fallback[[TEXT_COLUMN, LABEL_COLUMN]].dropna().drop_duplicates().reset_index(drop=True)
        train_df, test_df = train_test_split(
            fallback,
            test_size=0.20,
            random_state=seed,
            stratify=fallback[LABEL_COLUMN],
        )
        train_df, validation_df = train_test_split(
            train_df,
            test_size=TRAIN_VALIDATION_SPLIT,
            random_state=seed,
            stratify=train_df[LABEL_COLUMN],
        )
        source = f"local_fallback/{fallback_path.name}"

    return (
        train_df.reset_index(drop=True),
        validation_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
        source,
    )


train_df, validation_df, test_df, DATASET_SOURCE = load_imdb_splits(RANDOM_SEED)

print(f"Dataset source: {DATASET_SOURCE}")
print(f"Train samples: {len(train_df):,}")
print(f"Validation samples: {len(validation_df):,}")
print(f"Test samples: {len(test_df):,}")

## EDA

In [ ]:
def normalize_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def prepare_split(frame: pd.DataFrame) -> pd.DataFrame:
    prepared = frame.copy()
    prepared["raw_text"] = prepared[TEXT_COLUMN].astype(str)
    prepared["clean_text"] = prepared["raw_text"].map(normalize_text)
    prepared["char_count"] = prepared["raw_text"].str.len()
    prepared["token_count"] = prepared["clean_text"].str.split().map(len)
    prepared["sentiment"] = prepared[LABEL_COLUMN].map(LABEL_NAMES)
    return prepared


train_df = prepare_split(train_df)
validation_df = prepare_split(validation_df)
test_df = prepare_split(test_df)


def split_summary(frame: pd.DataFrame, split_name: str) -> dict:
    counts = frame[LABEL_COLUMN].value_counts().sort_index()
    return {
        "split": split_name,
        "samples": int(len(frame)),
        "negative": int(counts.get(0, 0)),
        "positive": int(counts.get(1, 0)),
        "negative_ratio": float(counts.get(0, 0) / len(frame)),
        "positive_ratio": float(counts.get(1, 0) / len(frame)),
        "avg_char_count": float(frame["char_count"].mean()),
        "median_char_count": float(frame["char_count"].median()),
        "avg_token_count": float(frame["token_count"].mean()),
        "median_token_count": float(frame["token_count"].median()),
    }


eda_summary = pd.DataFrame(
    [
        split_summary(train_df, "train"),
        split_summary(validation_df, "validation"),
        split_summary(test_df, "test"),
    ]
)

print("Split and length summary:")
display(eda_summary)

print("\nSample reviews from the train split:")
for label_value, label_name in LABEL_NAMES.items():
    samples = train_df.loc[train_df[LABEL_COLUMN] == label_value, ["raw_text", "token_count"]].head(2)
    print(f"\n{label_name.title()} examples:")
    for idx, (_, row) in enumerate(samples.iterrows(), start=1):
        print(f"[{idx}] tokens={int(row['token_count'])} :: {row['raw_text'][:350]}...")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
class_counts = train_df[LABEL_COLUMN].value_counts().sort_index()
sns.barplot(x=[LABEL_NAMES[0], LABEL_NAMES[1]], y=[class_counts.get(0, 0), class_counts.get(1, 0)], ax=axes[0], palette=["#355C7D", "#C06C84"])
axes[0].set_title("Train Class Balance")
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Count")

sns.histplot(train_df["token_count"], bins=40, kde=True, ax=axes[1], color="#6C5B7B")
axes[1].set_title("Train Token Length Distribution")
axes[1].set_xlabel("Token count")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig(EDA_DIR / "eda_class_balance_and_token_length.png", dpi=160, bbox_inches="tight")
plt.show()

## Text Processing and TF-IDF Feature Engineering

In [ ]:
def build_tfidf_vectorizer(config: dict) -> TfidfVectorizer:
    return TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        analyzer="word",
        token_pattern=r"(?u)\b\w\w+\b",
        dtype=np.float32,
        **config,
    )


tfidf_vectorizer = build_tfidf_vectorizer(TFIDF_CONFIG)
X_train = tfidf_vectorizer.fit_transform(train_df["clean_text"])
X_validation = tfidf_vectorizer.transform(validation_df["clean_text"])
X_test = tfidf_vectorizer.transform(test_df["clean_text"])

print(f"TF-IDF vocabulary size: {len(tfidf_vectorizer.vocabulary_):,}")
print(f"Train matrix shape: {X_train.shape}")
print(f"Validation matrix shape: {X_validation.shape}")
print(f"Test matrix shape: {X_test.shape}")

SPLIT_FRAMES = {"train": train_df, "validation": validation_df, "test": test_df}
SPLIT_MATRICES = {"train": X_train, "validation": X_validation, "test": X_test}

## Shared Utilities

In [ ]:
def to_serializable(value):
    if isinstance(value, dict):
        return {str(key): to_serializable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_serializable(item) for item in value]
    if isinstance(value, tuple):
        return [to_serializable(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value


def safe_write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(to_serializable(payload), handle, indent=2, ensure_ascii=False)


def safe_write_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def safe_save_numpy(path: Path, array) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, np.asarray(array))


def safe_save_model(path: Path, artifact) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(artifact, path)


def safe_save_figure(fig, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def extract_logits(estimator, probabilities, matrix):
    if hasattr(estimator, "predict_log_proba"):
        return estimator.predict_log_proba(matrix)
    if hasattr(estimator, "decision_function"):
        scores = estimator.decision_function(matrix)
        if np.ndim(scores) == 1:
            return np.column_stack([-scores, scores])
        return scores
    clipped = np.clip(probabilities, 1e-12, 1.0)
    return np.log(clipped)


def compute_metrics(y_true, y_pred, probabilities, inference_time_seconds: float) -> dict:
    confidence = probabilities.max(axis=1)
    counts = np.bincount(np.asarray(y_pred, dtype=int), minlength=2)
    total_predictions = int(counts.sum())
    majority_ratio = float(counts.max() / total_predictions) if total_predictions else 0.0
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "prediction_distribution": {str(index): int(count) for index, count in enumerate(counts)},
        "majority_class_ratio": majority_ratio,
        "confidence_mean": float(confidence.mean()),
        "confidence_std": float(confidence.std()),
        "confidence_min": float(confidence.min()),
        "confidence_median": float(np.median(confidence)),
        "confidence_max": float(confidence.max()),
        "inference_time_seconds": float(inference_time_seconds),
        "sample_count": int(len(y_true)),
    }


def build_prediction_frame(frame: pd.DataFrame, y_true, y_pred, probabilities, split_name: str) -> pd.DataFrame:
    probabilities = np.asarray(probabilities)
    confidence = probabilities.max(axis=1)
    positive_probability = probabilities[:, 1] if probabilities.shape[1] > 1 else probabilities[:, 0]
    negative_probability = probabilities[:, 0] if probabilities.shape[1] > 1 else 1.0 - probabilities[:, 0]
    result = frame[["raw_text", "clean_text", "sentiment", LABEL_COLUMN]].copy()
    result = result.rename(columns={"raw_text": "text", LABEL_COLUMN: "true_label", "sentiment": "true_sentiment"})
    result["split"] = split_name
    result["predicted_label"] = np.asarray(y_pred)
    result["predicted_sentiment"] = result["predicted_label"].map(LABEL_NAMES)
    result["confidence"] = confidence
    result["uncertainty"] = 1.0 - confidence
    result["is_correct"] = result["true_label"].to_numpy() == result["predicted_label"].to_numpy()
    result["negative_probability"] = negative_probability
    result["positive_probability"] = positive_probability
    result["prediction_margin"] = np.abs(positive_probability - negative_probability)
    return result


def build_error_analysis_frames(prediction_frame: pd.DataFrame, top_k: int = 50) -> dict:
    misclassified = prediction_frame.loc[~prediction_frame["is_correct"]].copy()
    most_uncertain = prediction_frame.sort_values(["confidence", "uncertainty"], ascending=[True, False]).head(top_k).copy()
    overconfident_wrong = misclassified.sort_values("confidence", ascending=False).head(top_k).copy()
    return {
        "misclassified": misclassified,
        "most_uncertain": most_uncertain,
        "overconfident_wrong": overconfident_wrong,
    }


def plot_confusion_matrix(confusion: np.ndarray, title: str, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        confusion,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[LABEL_NAMES[0], LABEL_NAMES[1]],
        yticklabels=[LABEL_NAMES[0], LABEL_NAMES[1]],
        cbar=False,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    safe_save_figure(fig, path)


def plot_confidence_histogram(prediction_frame: pd.DataFrame, title: str, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(
        data=prediction_frame,
        x="confidence",
        hue="is_correct",
        bins=30,
        kde=True,
        stat="density",
        common_norm=False,
        palette={True: "#2E7D32", False: "#C62828"},
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Confidence score")
    ax.set_ylabel("Density")
    ax.legend(title="Correct prediction", labels=["False", "True"])
    safe_save_figure(fig, path)


def plot_prediction_distribution(prediction_frame: pd.DataFrame, title: str, path: Path) -> None:
    counts = prediction_frame["predicted_sentiment"].value_counts().reindex([LABEL_NAMES[0], LABEL_NAMES[1]], fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(x=counts.index, y=counts.values, palette=["#355C7D", "#C06C84"], ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted sentiment")
    ax.set_ylabel("Count")
    for index, value in enumerate(counts.values):
        ax.text(index, value + max(counts.values) * 0.01, f"{value:,}", ha="center", va="bottom")
    safe_save_figure(fig, path)


def plot_metric_comparison(comparison_frame: pd.DataFrame, path: Path) -> None:
    plot_frame = comparison_frame.melt(id_vars="model", value_vars=["accuracy", "precision", "recall", "f1"], var_name="metric", value_name="score")
    fig, ax = plt.subplots(figsize=(11, 6))
    sns.barplot(data=plot_frame, x="metric", y="score", hue="model", ax=ax)
    ax.set_ylim(0, 1.0)
    ax.set_title("Classical Model Comparison on Test Split")
    ax.set_xlabel("Metric")
    ax.set_ylabel("Score")
    safe_save_figure(fig, path)


def flatten_model_config(config: dict) -> dict:
    flat = {}
    for key, value in config.items():
        if isinstance(value, dict):
            for sub_key, sub_value in value.items():
                flat[f"{key}_{sub_key}"] = sub_value
        else:
            flat[key] = value
    return flat


def build_classification_report_frame(report: dict) -> pd.DataFrame:
    return pd.DataFrame(report).T.reset_index().rename(columns={"index": "label"})


def safe_model_fitting(estimator, X_train, y_train, model_name: str):
    try:
        fit_start = perf_counter()
        estimator.fit(X_train, y_train)
        training_time_seconds = perf_counter() - fit_start
        print(f"Training completed in {training_time_seconds:.2f} seconds")
        return estimator, training_time_seconds
    except Exception as exc:
        raise RuntimeError(f"{model_name} training failed") from exc


def run_experiment(
    *,
    model_name: str,
    estimator,
    output_dir: Path,
    split_frames: dict,
    split_matrices: dict,
    vectorizer: TfidfVectorizer,
    global_config: dict,
    experiment_config: dict,
) -> dict:
    log_section(f"{model_name} - Training and evaluation")
    y_test = split_frames["test"][LABEL_COLUMN].to_numpy()

    estimator, training_time_seconds = safe_model_fitting(
        estimator,
        split_matrices["train"],
        split_frames["train"][LABEL_COLUMN].to_numpy(),
        model_name,
    )

    split_results = {}
    for split_name in ["train", "validation", "test"]:
        try:
            split_start = perf_counter()
            matrix = split_matrices[split_name]
            y_true = split_frames[split_name][LABEL_COLUMN].to_numpy()
            y_pred = estimator.predict(matrix)
            probabilities = estimator.predict_proba(matrix)
            inference_time_seconds = perf_counter() - split_start
            logits = extract_logits(estimator, probabilities, matrix)
            metrics = compute_metrics(y_true, y_pred, probabilities, inference_time_seconds)
            report = classification_report(
                y_true,
                y_pred,
                target_names=[LABEL_NAMES[0], LABEL_NAMES[1]],
                output_dict=True,
                zero_division=0,
            )
            confusion = confusion_matrix(y_true, y_pred, labels=[0, 1])
            prediction_frame = build_prediction_frame(split_frames[split_name], y_true, y_pred, probabilities, split_name)
            split_results[split_name] = {
                "y_true": y_true,
                "y_pred": y_pred,
                "probabilities": probabilities,
                "logits": logits,
                "metrics": metrics,
                "report": report,
                "confusion_matrix": confusion,
                "prediction_frame": prediction_frame,
                "inference_time_seconds": inference_time_seconds,
            }
            print(
                f"{split_name.title():>10}: accuracy={metrics['accuracy']:.4f}, f1={metrics['f1']:.4f}, inference={inference_time_seconds:.2f}s"
            )
        except Exception as exc:
            raise RuntimeError(f"{model_name} prediction or metric computation failed for split '{split_name}'") from exc

    test_prediction_frame = split_results["test"]["prediction_frame"].copy()
    error_frames = build_error_analysis_frames(test_prediction_frame)
    report_df = build_classification_report_frame(split_results["test"]["report"])
    confusion_df = pd.DataFrame(
        split_results["test"]["confusion_matrix"],
        index=[LABEL_NAMES[0], LABEL_NAMES[1]],
        columns=[LABEL_NAMES[0], LABEL_NAMES[1]],
    )

    convergence_status = "n/a"
    n_iter = None
    if hasattr(estimator, "n_iter_"):
        n_iter = int(np.max(np.asarray(estimator.n_iter_)))
        max_iter = int(experiment_config.get("model", {}).get("max_iter", 0))
        convergence_status = "converged" if max_iter == 0 or n_iter < max_iter else "max_iter_reached"

    history_row = {
        "model_name": model_name,
        "training_time_seconds": training_time_seconds,
        "train_inference_time_seconds": split_results["train"]["inference_time_seconds"],
        "validation_inference_time_seconds": split_results["validation"]["inference_time_seconds"],
        "test_inference_time_seconds": split_results["test"]["inference_time_seconds"],
        "train_accuracy": split_results["train"]["metrics"]["accuracy"],
        "validation_accuracy": split_results["validation"]["metrics"]["accuracy"],
        "train_precision": split_results["train"]["metrics"]["precision"],
        "validation_precision": split_results["validation"]["metrics"]["precision"],
        "train_recall": split_results["train"]["metrics"]["recall"],
        "validation_recall": split_results["validation"]["metrics"]["recall"],
        "train_f1": split_results["train"]["metrics"]["f1"],
        "validation_f1": split_results["validation"]["metrics"]["f1"],
        "test_accuracy": split_results["test"]["metrics"]["accuracy"],
        "test_precision": split_results["test"]["metrics"]["precision"],
        "test_recall": split_results["test"]["metrics"]["recall"],
        "test_f1": split_results["test"]["metrics"]["f1"],
        "convergence_status": convergence_status,
        "n_iter": n_iter,
        "vectorizer_vocab_size": len(vectorizer.vocabulary_),
        **flatten_model_config(experiment_config),
    }

    train_metrics = {
        "model": model_name,
        "training_time_seconds": training_time_seconds,
        "train": split_results["train"]["metrics"],
        "validation": split_results["validation"]["metrics"],
        "train_inference_time_seconds": split_results["train"]["inference_time_seconds"],
        "validation_inference_time_seconds": split_results["validation"]["inference_time_seconds"],
        "experiment_config": experiment_config,
    }
    test_metrics = {
        "model": model_name,
        "metrics": split_results["test"]["metrics"],
        "inference_time_seconds": split_results["test"]["inference_time_seconds"],
        "training_time_seconds": training_time_seconds,
        "sample_count": int(len(y_test)),
    }

    safe_write_json(output_dir / "classification_report.json", {"model": model_name, "report": split_results["test"]["report"]})
    safe_write_csv(report_df, output_dir / "classification_report.csv")
    safe_write_csv(confusion_df.reset_index().rename(columns={"index": "label"}), output_dir / "confusion_matrix.csv")
    safe_write_json(output_dir / "config.json", global_config)
    safe_write_json(output_dir / "experiment_config.json", experiment_config)
    safe_save_numpy(output_dir / "labels.npy", y_test)
    safe_save_numpy(output_dir / "logits.npy", split_results["test"]["logits"])
    safe_save_numpy(output_dir / "predictions.npy", split_results["test"]["y_pred"])
    safe_save_numpy(output_dir / "probabilities.npy", split_results["test"]["probabilities"])
    safe_write_csv(test_prediction_frame, output_dir / "prediction_analysis.csv")
    safe_write_json(output_dir / "test_metrics.json", test_metrics)
    safe_write_json(output_dir / "train_metrics.json", train_metrics)
    safe_write_csv(pd.DataFrame([history_row]), output_dir / "training_history.csv")
    safe_save_model(output_dir / "model.pkl", estimator)
    safe_save_model(output_dir / "vectorizer.pkl", vectorizer)
    safe_write_csv(error_frames["misclassified"], output_dir / "misclassified_samples.csv")
    safe_write_csv(error_frames["most_uncertain"], output_dir / "most_uncertain_samples.csv")
    safe_write_csv(error_frames["overconfident_wrong"], output_dir / "overconfident_wrong_predictions.csv")

    plot_confusion_matrix(split_results["test"]["confusion_matrix"], f"{model_name} Confusion Matrix", output_dir / "confusion_matrix.png")
    plot_confidence_histogram(test_prediction_frame, f"{model_name} Confidence Distribution", output_dir / "confidence_histogram.png")
    plot_prediction_distribution(test_prediction_frame, f"{model_name} Prediction Distribution", output_dir / "prediction_distribution.png")

    print(f"Saved artifacts to: {output_dir}")
    print(f"Misclassified samples: {len(error_frames['misclassified']):,}")
    print(f"Most uncertain samples: {len(error_frames['most_uncertain']):,}")
    print(f"Most overconfident wrong predictions: {len(error_frames['overconfident_wrong']):,}")

    return {
        "model_name": model_name,
        "estimator": estimator,
        "output_dir": output_dir,
        "split_results": split_results,
        "prediction_analysis": test_prediction_frame,
        "error_frames": error_frames,
        "history_row": history_row,
        "train_metrics": train_metrics,
        "test_metrics": test_metrics,
        "report_df": report_df,
        "confusion_df": confusion_df,
    }

## Naive Bayes Pipeline

In [ ]:
naive_bayes_global_config = {
    "artifact_schema_version": ARTIFACT_SCHEMA_VERSION,
    "project_title": PROJECT_TITLE,
    "dataset": {
        "name": DATASET_NAME,
        "source": DATASET_SOURCE,
        "train_size": int(len(train_df)),
        "validation_size": int(len(validation_df)),
        "test_size": int(len(test_df)),
        "random_seed": RANDOM_SEED,
        "label_names": LABEL_NAMES,
    },
    "tfidf": TFIDF_CONFIG,
    "evaluation": {
        "metrics": ["accuracy", "precision", "recall", "f1"],
        "zero_division": 0,
        "confidence_definition": "max predicted probability",
    },
}
naive_bayes_experiment_config = {
    "model": {
        "name": "MultinomialNB",
        **NAIVE_BAYES_CONFIG,
    },
    "tfidf": TFIDF_CONFIG,
}

naive_bayes_model = MultinomialNB(**NAIVE_BAYES_CONFIG)
naive_bayes_results = run_experiment(
    model_name="Naive Bayes",
    estimator=naive_bayes_model,
    output_dir=NAIVE_BAYES_DIR,
    split_frames=SPLIT_FRAMES,
    split_matrices=SPLIT_MATRICES,
    vectorizer=tfidf_vectorizer,
    global_config=naive_bayes_global_config,
    experiment_config=naive_bayes_experiment_config,
)

display(pd.DataFrame([naive_bayes_results["test_metrics"]["metrics"]]).T.rename(columns={0: "value"}))

## Logistic Regression Pipeline

In [ ]:
logistic_regression_global_config = naive_bayes_global_config
logistic_regression_experiment_config = {
    "model": {
        "name": "LogisticRegression",
        **LOGISTIC_REGRESSION_CONFIG,
    },
    "tfidf": TFIDF_CONFIG,
}

logistic_regression_model = LogisticRegression(
    random_state=RANDOM_SEED,
    **LOGISTIC_REGRESSION_CONFIG,
)
logistic_regression_results = run_experiment(
    model_name="Logistic Regression",
    estimator=logistic_regression_model,
    output_dir=LOGISTIC_REGRESSION_DIR,
    split_frames=SPLIT_FRAMES,
    split_matrices=SPLIT_MATRICES,
    vectorizer=tfidf_vectorizer,
    global_config=logistic_regression_global_config,
    experiment_config=logistic_regression_experiment_config,
)

display(pd.DataFrame([logistic_regression_results["test_metrics"]["metrics"]]).T.rename(columns={0: "value"}))

## Evaluation

In [ ]:
evaluation_rows = []
for result in [naive_bayes_results, logistic_regression_results]:
    test_metrics = result["test_metrics"]["metrics"]
    evaluation_rows.append(
        {
            "model": result["model_name"],
            "accuracy": test_metrics["accuracy"],
            "precision": test_metrics["precision"],
            "recall": test_metrics["recall"],
            "f1": test_metrics["f1"],
            "train_time": result["history_row"]["training_time_seconds"],
            "inference_time": test_metrics["inference_time_seconds"],
            "majority_class_ratio": test_metrics["majority_class_ratio"],
            "confidence_mean": test_metrics["confidence_mean"],
            "confidence_std": test_metrics["confidence_std"],
        }
    )

comparison_df = pd.DataFrame(evaluation_rows).sort_values(["f1", "accuracy"], ascending=False).reset_index(drop=True)
print("Test-set comparison table:")
display(comparison_df)

train_validation_rows = []
for result in [naive_bayes_results, logistic_regression_results]:
    train_validation_rows.append(
        {
            "model": result["model_name"],
            "train_accuracy": result["split_results"]["train"]["metrics"]["accuracy"],
            "validation_accuracy": result["split_results"]["validation"]["metrics"]["accuracy"],
            "train_f1": result["split_results"]["train"]["metrics"]["f1"],
            "validation_f1": result["split_results"]["validation"]["metrics"]["f1"],
            "train_precision": result["split_results"]["train"]["metrics"]["precision"],
            "validation_precision": result["split_results"]["validation"]["metrics"]["precision"],
            "train_recall": result["split_results"]["train"]["metrics"]["recall"],
            "validation_recall": result["split_results"]["validation"]["metrics"]["recall"],
        }
    )

train_validation_df = pd.DataFrame(train_validation_rows)
print("Train/validation summary:")
display(train_validation_df)

## Error Analysis

In [ ]:
for result in [naive_bayes_results, logistic_regression_results]:
    print("\n" + "-" * 100)
    print(f"{result['model_name']} error analysis")
    print("-" * 100)
    misclassified = result["error_frames"]["misclassified"]
    uncertain = result["error_frames"]["most_uncertain"]
    wrong_confident = result["error_frames"]["overconfident_wrong"]
    print(f"Misclassified samples: {len(misclassified):,}")
    print(f"Most uncertain confidence range: {uncertain['confidence'].min():.4f} to {uncertain['confidence'].max():.4f}")
    print(f"Most overconfident wrong confidence range: {wrong_confident['confidence'].min():.4f} to {wrong_confident['confidence'].max():.4f}")
    display(uncertain[["text", "true_sentiment", "predicted_sentiment", "confidence"]].head(5))

## Artifact Saving

In [ ]:
required_artifacts = [
    "classification_report.json",
    "classification_report.csv",
    "config.json",
    "experiment_config.json",
    "labels.npy",
    "logits.npy",
    "predictions.npy",
    "probabilities.npy",
    "prediction_analysis.csv",
    "test_metrics.json",
    "train_metrics.json",
    "training_history.csv",
    "model.pkl",
    "vectorizer.pkl",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "confidence_histogram.png",
    "prediction_distribution.png",
    "misclassified_samples.csv",
    "most_uncertain_samples.csv",
    "overconfident_wrong_predictions.csv",
]

validation_rows = []
for model_name, model_dir in [("Naive Bayes", naive_bayes_results["output_dir"]), ("Logistic Regression", logistic_regression_results["output_dir"])]:
    for artifact in required_artifacts:
        validation_rows.append(
            {
                "model": model_name,
                "artifact": artifact,
                "exists": (model_dir / artifact).exists(),
            }
        )

artifact_validation_df = pd.DataFrame(validation_rows)
display(artifact_validation_df.pivot(index="artifact", columns="model", values="exists"))

## Comparative Summary

In [ ]:
combined_metrics_df = comparison_df.copy()
combined_metrics_df["train_accuracy"] = [
    naive_bayes_results["split_results"]["train"]["metrics"]["accuracy"],
    logistic_regression_results["split_results"]["train"]["metrics"]["accuracy"],
]
combined_metrics_df["validation_accuracy"] = [
    naive_bayes_results["split_results"]["validation"]["metrics"]["accuracy"],
    logistic_regression_results["split_results"]["validation"]["metrics"]["accuracy"],
]
combined_metrics_df["prediction_distribution"] = [
    naive_bayes_results["split_results"]["test"]["metrics"]["prediction_distribution"],
    logistic_regression_results["split_results"]["test"]["metrics"]["prediction_distribution"],
]

safe_write_csv(combined_metrics_df, PROJECT_ROOT / "combined_metrics.csv")
safe_write_json(PROJECT_ROOT / "combined_metrics.json", combined_metrics_df.to_dict(orient="records"))
plot_metric_comparison(combined_metrics_df, PROJECT_ROOT / "combined_metric_comparison.png")

print("Combined metrics:")
display(combined_metrics_df)

best_model = combined_metrics_df.iloc[0]["model"]
print(f"\nBest model by test F1-score: {best_model}")
print("Artifacts written to:")
print(f" - {naive_bayes_results['output_dir']}")
print(f" - {logistic_regression_results['output_dir']}")
print(f" - {PROJECT_ROOT / 'combined_metrics.csv'}")

## Conclusion

The classical baseline now exports a standardized artifact bundle that is aligned with the transformer comparison workflow.